In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["tweets_db"]
collection = db["airline_tweets"]

print("✅ Connected to MongoDB!")
print("Total documents:", collection.count_documents({}))

✅ Connected to MongoDB!
Total documents: 14478


In [2]:
print("=== Sentiment Count ===")
pipeline1 = [
    {"$group": {"_id": "$airline_sentiment", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
for doc in collection.aggregate(pipeline1):
    print(doc)

=== Sentiment Count ===
{'_id': 'negative', 'count': 9075}
{'_id': 'neutral', 'count': 3069}
{'_id': 'positive', 'count': 2334}


In [4]:
print("\n=== Sentiment by Airline ===")
pipeline2 = [
    {"$group": {
        "_id": {
            "airline": "$airline",
            "sentiment": "$airline_sentiment"
        },
        "count": {"$sum": 1}
    }},
    {"$sort": {"_id.airline": 1, "count": -1}}
]
for doc in collection.aggregate(pipeline2):
    print(doc)


=== Sentiment by Airline ===
{'_id': {'airline': 'American', 'sentiment': 'negative'}, 'count': 1863}
{'_id': {'airline': 'American', 'sentiment': 'neutral'}, 'count': 433}
{'_id': {'airline': 'American', 'sentiment': 'positive'}, 'count': 307}
{'_id': {'airline': 'Delta', 'sentiment': 'negative'}, 'count': 953}
{'_id': {'airline': 'Delta', 'sentiment': 'neutral'}, 'count': 723}
{'_id': {'airline': 'Delta', 'sentiment': 'positive'}, 'count': 544}
{'_id': {'airline': 'Southwest', 'sentiment': 'negative'}, 'count': 1185}
{'_id': {'airline': 'Southwest', 'sentiment': 'neutral'}, 'count': 664}
{'_id': {'airline': 'Southwest', 'sentiment': 'positive'}, 'count': 570}
{'_id': {'airline': 'US Airways', 'sentiment': 'negative'}, 'count': 2263}
{'_id': {'airline': 'US Airways', 'sentiment': 'neutral'}, 'count': 381}
{'_id': {'airline': 'US Airways', 'sentiment': 'positive'}, 'count': 269}
{'_id': {'airline': 'United', 'sentiment': 'negative'}, 'count': 2630}
{'_id': {'airline': 'United', 'senti

In [5]:
print("\n=== Top Negative Reasons ===")
pipeline3 = [
    {"$match": {
        "airline_sentiment": "negative",
        "negativereason": {"$ne": None}
    }},
    {"$group": {"_id": "$negativereason", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10}
]
for doc in collection.aggregate(pipeline3):
    print(doc)


=== Top Negative Reasons ===
{'_id': 'Customer Service Issue', 'count': 2883}
{'_id': 'Late Flight', 'count': 1648}
{'_id': "Can't Tell", 'count': 1175}
{'_id': 'Cancelled Flight', 'count': 829}
{'_id': 'Lost Luggage', 'count': 717}
{'_id': 'Bad Flight', 'count': 575}
{'_id': 'Flight Booking Problems', 'count': 523}
{'_id': 'Flight Attendant Complaints', 'count': 475}
{'_id': 'longlines', 'count': 177}
{'_id': 'Damaged Luggage', 'count': 73}


In [6]:
import time

# Query with index (airline_sentiment is indexed)
start = time.time()
list(collection.find({"airline_sentiment": "negative"}))
end = time.time()
print(f"⚡ Query WITH index time:  {end - start:.4f} sec")

# Show all indexes
print("\n✅ Active indexes:")
for idx in collection.index_information():
    print("  -", idx)

⚡ Query WITH index time:  0.0745 sec

✅ Active indexes:
  - _id_
  - airline_1
  - airline_sentiment_1
  - negativereason_1
  - airline_1_airline_sentiment_1
  - tweet_id_1


In [7]:
print("\n=== Most Complained Airline ===")
pipeline4 = [
    {"$match": {"airline_sentiment": "negative"}},
    {"$group": {"_id": "$airline", "complaints": {"$sum": 1}}},
    {"$sort": {"complaints": -1}},
    {"$limit": 3}
]
for doc in collection.aggregate(pipeline4):
    print(doc)


=== Most Complained Airline ===
{'_id': 'United', 'complaints': 2630}
{'_id': 'US Airways', 'complaints': 2263}
{'_id': 'American', 'complaints': 1863}


In [8]:
print("\n=== Most Praised Airline ===")
pipeline5 = [
    {"$match": {"airline_sentiment": "positive"}},
    {"$group": {"_id": "$airline", "praise": {"$sum": 1}}},
    {"$sort": {"praise": -1}},
    {"$limit": 3}
]
for doc in collection.aggregate(pipeline5):
    print(doc)


=== Most Praised Airline ===
{'_id': 'Southwest', 'praise': 570}
{'_id': 'Delta', 'praise': 544}
{'_id': 'United', 'praise': 492}


In [9]:
# Find worst airline name from pipeline4 result
worst = list(collection.aggregate([
    {"$match": {"airline_sentiment": "negative"}},
    {"$group": {"_id": "$airline", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 1}
]))[0]["_id"]

print(f"\n=== Sample Negative Tweets from {worst} ===")
results = collection.find(
    {"airline": worst, "airline_sentiment": "negative"},
    {"text": 1, "negativereason": 1, "_id": 0}
).limit(5)
for doc in results:
    print(doc)


=== Sample Negative Tweets from United ===
{'negativereason': 'Cancelled Flight', 'text': "@united still no refund or word via DM. Please resolve this issue as your Cancelled Flightled flight was useless to my assistant's trip."}
{'negativereason': 'Late Flight', 'text': "@united Delayed due to lack of crew and now delayed again because there's a long line for deicing... Still need to improve service #united"}
{'negativereason': "Can't Tell", 'text': '@united Your ERI-ORD express connections are hugely popular .. now if only we could have an ERI-EWR hop! :)'}
{'negativereason': "Can't Tell", 'text': '@united you think you boarded flight AU1066 too early? I think so.'}
{'negativereason': 'Bad Flight', 'text': '@united Gate agent hooked me up with alternate flights. If you have a way to PREVENT the constant issues, that would rock.'}
